# Project 3: Unsupervised Learning - Customer Segmentation

## Goal

The goal of this project is to group customers based on their purchasing behavior.

Since the data does not have predefined customer groups, we will use clustering to discover patterns in the data and create useful customer segments.

## Objectives

### 1. Prepare and Scale the Data

We will select the customer features that will be used for clustering.

Since the features may have different scales, we will use `StandardScaler` to standardize them. This makes sure that one feature does not have more influence on the clustering just because it has larger values.

### 2. Apply PCA

We will use Principal Component Analysis (PCA) to reduce the number of features while keeping as much useful information as possible.

PCA will also help us visualize the customer groups more easily.

### 3. Find the Best Number of Clusters

We will use two methods to determine the appropriate number of customer groups:

- **Elbow Method:** Helps us see where adding more clusters stops giving a major improvement.
- **Silhouette Score:** Measures how well customers fit within their assigned clusters.

We will compare the results from both methods before choosing the number of clusters.

### 4. Build Customer Clusters

After choosing the best value of K, we will apply **K-Means Clustering** to group customers with similar purchasing behavior.

Each customer will be assigned to one cluster.

### 5. Understand the Customer Groups

We will examine the characteristics of each cluster to understand how the groups differ.

For example, some groups may represent:

- High-value customers
- Frequent buyers
- Occasional buyers
- Low-spending customers

The actual personas will be based on the patterns found in the data.

### 6. Create Business Personas

Finally, we will translate the cluster results into simple customer personas.

We will look at the original customer features and cluster averages to describe each group in business terms.

These personas can help the business understand its customers and make better decisions about marketing, promotions, and customer engagement.

In [1]:
import pandas as pd

# Load the data
df = pd.read_csv('Dataset for Data Analytics - Sheet1.csv')

# Replace missing coupon codes
df['CouponCode'] = df['CouponCode'].fillna('NO_COUPON')

# Group data by customer
customer_df = df.groupby('CustomerID').agg({
    'TotalPrice': 'sum',
    'OrderID': 'count',
    'Quantity': 'mean',
    'UnitPrice': 'mean',
    'ItemsInCart': 'mean'
}).reset_index()

# Calculate coupon usage
coupon_use = df.groupby('CustomerID')['CouponCode'].apply(
    lambda x: (x != 'NO_COUPON').mean()
).reset_index(name='DiscountUsedRatio')

# Add coupon usage to customer data
customer_df = customer_df.merge(coupon_use, on='CustomerID')

# Rename columns
customer_df.columns = [
    'CustomerID',
    'TotalSpend',
    'TotalOrders',
    'AvgQuantity',
    'AvgUnitPrice',
    'AvgItemsInCart',
    'DiscountUsedRatio'
]

customer_df.head()

,CustomerID,TotalSpend,TotalOrders,AvgQuantity,AvgUnitPrice,AvgItemsInCart,DiscountUsedRatio
0,C10002,1470.03,1,3.0,490.01,4.0,1.0
1,C10054,153.48,1,2.0,76.74,7.0,1.0
2,C10126,1349.80,1,2.0,674.90,4.0,1.0
3,C10154,1303.04,1,4.0,325.76,8.0,1.0
4,C10211,635.90,1,5.0,127.18,10.0,1.0


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Select features
features = [
    'TotalSpend',
    'TotalOrders',
    'AvgQuantity',
    'AvgUnitPrice',
    'AvgItemsInCart',
    'DiscountUsedRatio'
]

# Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(customer_df[features])

# Reduce to 3 principal components
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

# Check the result
X_pca[:5]

array([[ 0.24353749, -1.00934275, -0.33642008],
       [-1.22260344,  1.29432378, -0.33592531],
       [ 0.09915565, -1.91028409, -0.39201283],
       [ 1.03574   ,  0.7470129 , -0.38553096],
       [ 0.99049137,  2.39532637, -0.41164511]])

In [3]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

wcss = []
silhouette_scores = []

for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_pca)

    wcss.append(model.inertia_)
    silhouette_scores.append(silhouette_score(X_pca, labels))

print("WCSS:", wcss)
print("Silhouette Scores:", silhouette_scores)

WCSS: [3569.7269723506897, 2610.0270855913573, 1746.273330123075, 1343.8916010057083, 1143.2573315636575, 1018.1762111931549, 902.3204699776702]
Silhouette Scores: [0.35348460700902623, 0.36997444994067646, 0.37626588024878777, 0.35725907520074845, 0.333975985861621, 0.314758635720694, 0.31482090559215287]


In [5]:
from sklearn.cluster import KMeans

# Create 4 customer groups
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
customer_df['Cluster'] = kmeans.fit_predict(X_pca)

# Summarize each customer group
persona_matrix = customer_df.groupby('Cluster').agg({
    'CustomerID': 'count',
    'TotalSpend': 'mean',
    'TotalOrders': 'mean',
    'AvgQuantity': 'mean',
    'AvgUnitPrice': 'mean',
    'AvgItemsInCart': 'mean',
    'DiscountUsedRatio': 'mean'
}).round(2)

persona_matrix

,CustomerID,TotalSpend,TotalOrders,AvgQuantity,AvgUnitPrice,AvgItemsInCart,DiscountUsedRatio
Cluster,,,,,,,
0,330,700.52,1.0,3.87,182.63,6.91,0.65
1,337,2143.67,1.0,4.03,537.57,6.81,0.78
2,11,1775.98,2.0,2.73,317.79,5.05,0.73
3,511,570.72,1.0,1.65,350.83,3.71,0.78


# Conclusion

This project used customer transaction data to identify different groups of customers based on their purchasing behavior.

The data was first standardized using `StandardScaler` so that all features could be compared fairly. PCA was then used to reduce the number of features and make the data easier to work with.

K-Means clustering was tested with different numbers of clusters using the Elbow Method and Silhouette Score. The results were used to select 4 customer groups.

The final clusters showed that customers have different spending, ordering, purchasing, and discount usage patterns. These groups can be used to create simple customer personas and help the business understand its customers better.

The customer segments can support decisions such as targeted marketing, promotions, customer retention, and personalized offers.